# CLLP-32: reachability across three Sokoban families

Real, separately verified GPU runs. The model was trained on 12×12×4; 15×15×10 is a strong out-of-distribution transfer test. k is a witness-action offset, not shortest-path distance.


In [ ]:
from pathlib import Path
import os, json, csv
import numpy as np
from PIL import Image
from IPython.display import display
from carl.icl.boards import from_rows, to_rows, state_hash
from carl.icl.data_sources.sokoban import environment
here=Path.cwd().resolve();repo=next((p for p in (here,*here.parents) if (p/'analysis/torch_reachability').is_dir()),None)
if repo is None:raise RuntimeError('Run inside the CaRL checkout.')
base=Path(os.environ.get('CARL_DENSE_GPU_BASE',repo/'analysis/torch_reachability/dense_cllp32_gpu_v1')).resolve()
new=Path(os.environ.get('CARL_DENSE15_GPU_RESULTS',repo/'analysis/torch_reachability/dense15_cllp32_gpu_v1')).resolve()
for d in (base,new):
    verification=d/'export/verification.json'
    if not verification.is_file():raise RuntimeError(f'Missing verified export: {verification}')
    result=json.loads(verification.read_text());print(d.name, json.dumps(result['coverage'],indent=2))
    if not result.get('all_present_traces_replayed') or not result.get('complete'):raise RuntimeError(f'Incomplete or unverified export: {d}')
for name in ('summary_all_families.csv','figure1_three_families.png','first_hit_three_families.png'):
    if not (new/'comparison'/name).is_file():raise RuntimeError(f'Missing comparison {name}; run dense15_report first')
print('12×12 contract:',json.dumps(json.loads((base/'contract.json').read_text()),indent=2));print('15×15 contract:',json.dumps(json.loads((new/'contract.json').read_text()),indent=2))


## Reachability and first-hit time

Bootstrap intervals resample whole layouts; three anchors on one layout are not three independent boards. Different families contain different layouts.


In [ ]:
with (new/'comparison/summary_all_families.csv').open() as f:summary=list(csv.DictReader(f))
for row in summary:print(row)
display(Image.open(new/'comparison/figure1_three_families.png'));display(Image.open(new/'comparison/first_hit_three_families.png'))


## Evaluation layouts

These are initial states; the tested source states can occur later in each witness.


In [ ]:
names={'#':'wall.png','-':'floor.png','.':'box_target.png','$':'box.png','*':'box_on_target.png','@':'player.png','+':'player_on_target.png'}
sprites={k:np.asarray(Image.open(repo/'carl/environment/sokoban/surface/16x16pixels'/v).convert('RGB')) for k,v in names.items()}
def board_image(rows,size=120):
    pixels=np.concatenate([np.concatenate([sprites[c] for c in row],axis=1) for row in rows],axis=0)
    return Image.fromarray(pixels).resize((size,size),Image.Resampling.NEAREST)
for family,d in (('12x4',base),('12x10',base),('15x10',new)):
    manifest=json.loads((d/family/'manifest.json').read_text());sheet=Image.new('RGB',(10*120,5*120),'white')
    for i,b in enumerate(manifest['boards']):sheet.paste(board_image(b['initial_state']),((i%10)*120,(i//10)*120))
    print(family,len(manifest['boards']),'layouts');display(sheet)


## A replayed 15×15×10 success

The exact source, target and each saved action are reconstructed in the simulator.


In [ ]:
with (new/'export/trials.csv').open() as f:trials=list(csv.DictReader(f))
examples=[r for r in trials if r['status']=='complete' and r['exact_hit']=='True' and r['first_hit'] and int(r['first_hit'])>=4]
if not examples:print('No qualifying successful trace in this complete export.')
else:
    sample=examples[0];print('Trial:',sample)
    manifest=json.loads((new/'15x10/manifest.json').read_text());b=next(x for x in manifest['boards'] if x['layout_id']==sample['layout_id'])
    rec=json.loads((new/sample['trace_file']).read_text());s=from_rows(b['initial_state']);env=environment(s);states=[s]
    for action in b['actions']:states.append(np.asarray(env.next_state(states[-1],action)))
    current=states[rec['i']];goal=states[rec['i']+rec['k']]
    print('Source');display(board_image(to_rows(current),300));print('Target');display(board_image(to_rows(goal),300))
    for j,action in enumerate(rec['actions'],1):
        assert state_hash(current)==rec['state_hashes'][j-1];current=np.asarray(env.next_state(current,action));assert state_hash(current)==rec['state_hashes'][j]
        print(j,['UP','DOWN','LEFT','RIGHT'][action]);display(board_image(to_rows(current),300))
    assert state_hash(current)==rec['target_hash']
